# Reference: Human Feedback in MLflow 3

---

This notebook is a reference guide to MLflow's feedback and expectation APIs — the system for attaching quality assessments and ground truth to traces.

**What this covers:**

| Capability | API | Section |
|---|---|---|
| Log feedback on a trace | `mlflow.log_feedback()` | §1 |
| Feedback value types | `bool`, `float`, `str`, `dict` | §2 |
| Assessment sources | `HUMAN`, `CODE`, `LLM_JUDGE` | §3 |
| Log ground-truth expectations | `mlflow.log_expectation()` | §4 |
| Retrieve assessments | `mlflow.get_assessment()` | §5 |
| Update assessments | `mlflow.update_assessment()` | §5 |
| Delete assessments | `mlflow.delete_assessment()` | §5 |
| Override automated feedback | `mlflow.override_feedback()` | §6 |
| Batch annotation | Helper patterns | §7 |
| Span-level feedback | `span_id` parameter | §8 |

> **Requirements:** MLflow 3.2+, a tracking server with a SQL backend (SQLite, PostgreSQL, etc.), and at least one logged trace.

---
## 0 — Setup

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import mlflow
from mlflow.entities import AssessmentSource
from mlflow.entities.assessment_source import AssessmentSourceType

load_dotenv()

tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("feedback-reference")
mlflow.openai.autolog()

client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

print(f"MLflow version: {mlflow.__version__}")

### Generate some traces to annotate

We need traces to attach feedback to. Let's create a few.

In [ ]:
@mlflow.trace
def ask(question: str) -> str:
    response = client.chat.completions.create(
        model="gemini-3.1-flash-lite-preview",
        messages=[{"role": "user", "content": question}],
        max_completion_tokens=256,
    )
    return response.choices[0].message.content


# Generate traces and capture their IDs
questions = [
    "What is the capital of France?",
    "Explain photosynthesis in one paragraph.",
    "What are the three laws of thermodynamics?",
]

trace_ids = []
for q in questions:
    answer = ask(q)
    trace = mlflow.get_last_active_trace()
    trace_ids.append(trace.info.trace_id)
    print(f"Trace {trace.info.trace_id[:16]}...  Q: {q}")

print(f"\nGenerated {len(trace_ids)} traces.")

---
## 1 — Logging Feedback with `mlflow.log_feedback()`

Feedback is a quality assessment attached to a trace. The core function is `mlflow.log_feedback()`:

```python
mlflow.log_feedback(
    trace_id="...",           # which trace to annotate
    name="...",               # what quality dimension (e.g. "correctness", "helpfulness")
    value=...,                # the assessment (bool, float, str, dict)
    rationale="...",          # why this assessment was given
    source=AssessmentSource(  # who gave this feedback
        source_type=AssessmentSourceType.HUMAN,
        source_id="reviewer@company.com",
    ),
)
```

In [ ]:
# Thumbs up/down on the first trace
mlflow.log_feedback(
    trace_id=trace_ids[0],
    name="user_satisfaction",
    value=True,
    rationale="Response correctly identified Paris as the capital of France.",
    source=AssessmentSource(
        source_type=AssessmentSourceType.HUMAN,
        source_id="reviewer@example.com",
    ),
)

print(f"Feedback logged on trace {trace_ids[0][:16]}...")

---
## 2 — Feedback Value Types

The `value` parameter accepts multiple types depending on what you're measuring:

| Type | Use case | Example |
|---|---|---|
| `bool` | Binary judgments (thumbs up/down, pass/fail) | `True`, `False` |
| `float` | Scores on a scale | `0.85`, `4.5` |
| `int` | Star ratings, discrete scores | `5`, `3` |
| `str` | Categorical labels | `"excellent"`, `"needs_improvement"` |
| `dict` | Structured multi-dimensional feedback | `{"accuracy": 0.9, "tone": "professional"}` |

In [ ]:
human_source = AssessmentSource(
    source_type=AssessmentSourceType.HUMAN,
    source_id="reviewer@example.com",
)

# Boolean — pass/fail
mlflow.log_feedback(
    trace_id=trace_ids[0],
    name="is_factually_correct",
    value=True,
    rationale="Paris is indeed the capital of France.",
    source=human_source,
)

# Float — score on a scale
mlflow.log_feedback(
    trace_id=trace_ids[1],
    name="completeness",
    value=0.75,
    rationale="Covers the basics of photosynthesis but misses the Calvin cycle.",
    source=human_source,
)

# String — categorical label
mlflow.log_feedback(
    trace_id=trace_ids[1],
    name="quality_tier",
    value="good",
    rationale="Accurate and well-structured, minor gaps.",
    source=human_source,
)

# Integer — star rating
mlflow.log_feedback(
    trace_id=trace_ids[2],
    name="star_rating",
    value=4,
    rationale="Good explanation of all three laws but could be more concise.",
    source=human_source,
)

print("Logged feedback with bool, float, str, and int values.")

In [ ]:
# Structured — multi-dimensional assessment in a single feedback entry
mlflow.log_feedback(
    trace_id=trace_ids[2],
    name="detailed_review",
    value={
        "accuracy": 0.9,
        "clarity": 0.8,
        "conciseness": 0.6,
        "missing_topics": ["entropy examples", "real-world applications"],
    },
    rationale="Technically accurate but verbose. Missing practical examples.",
    source=human_source,
)

print("Logged structured feedback.")

---
## 3 — Assessment Sources

Every piece of feedback tracks *who* or *what* produced it. MLflow supports three source types:

| Source type | `source_id` convention | When to use |
|---|---|---|
| `HUMAN` | Email, username, or reviewer ID | Manual reviews, user thumbs up/down |
| `CODE` | Script name, rule ID, or checker version | Deterministic checks (regex, length, format) |
| `LLM_JUDGE` | Model identifier (e.g. `openai:/gpt-4o`) | Automated LLM-based evaluation |

The source lets you filter, compare, and weight feedback appropriately — for example, human feedback might override an LLM judge's assessment.

In [ ]:
# Human reviewer
human_source = AssessmentSource(
    source_type=AssessmentSourceType.HUMAN,
    source_id="alice@example.com",
)

# Programmatic rule
code_source = AssessmentSource(
    source_type=AssessmentSourceType.CODE,
    source_id="length_checker_v1",
)

# LLM judge
llm_source = AssessmentSource(
    source_type=AssessmentSourceType.LLM_JUDGE,
    source_id="openai:/gpt-4o-mini",
)

# Same trace, three different sources
mlflow.log_feedback(
    trace_id=trace_ids[0],
    name="correctness_human",
    value=True,
    rationale="Verified by domain expert.",
    source=human_source,
)

mlflow.log_feedback(
    trace_id=trace_ids[0],
    name="response_length_check",
    value=True,
    rationale="Response is under 500 characters.",
    source=code_source,
)

mlflow.log_feedback(
    trace_id=trace_ids[0],
    name="correctness_judge",
    value=True,
    rationale="The response accurately states that Paris is the capital of France.",
    source=llm_source,
)

print("Logged feedback from three different source types on the same trace.")

---
## 4 — Ground-Truth Expectations with `mlflow.log_expectation()`

Expectations are different from feedback. **Feedback** evaluates what the model *did*. **Expectations** define what the model *should have done* — the ground truth.

| | Feedback | Expectation |
|---|---|---|
| **Answers** | "Was the output good?" | "What should the output have been?" |
| **Timing** | After reviewing the output | Can be set before or after |
| **Use case** | Quality scoring | Building eval datasets, regression testing |

Expectations attach to traces the same way feedback does, and can later be pulled into evaluation datasets.

In [ ]:
expert_source = AssessmentSource(
    source_type=AssessmentSourceType.HUMAN,
    source_id="subject_expert@example.com",
)

# Simple factual expectation
mlflow.log_expectation(
    trace_id=trace_ids[0],
    name="expected_answer",
    value="The capital of France is Paris.",
    source=expert_source,
)

print("Logged factual expectation.")

In [ ]:
# Structured expectation — multiple required elements
mlflow.log_expectation(
    trace_id=trace_ids[1],
    name="expected_facts",
    value=[
        "light energy",
        "carbon dioxide",
        "water",
        "glucose",
        "oxygen",
        "chlorophyll",
    ],
    source=expert_source,
)

print("Logged structured expectation (list of required facts).")

In [ ]:
# Behavioral expectation — how the model should behave, not just what it says
mlflow.log_expectation(
    trace_id=trace_ids[2],
    name="expected_behavior",
    value={
        "should_number_laws": True,
        "should_give_examples": True,
        "max_paragraphs": 3,
        "tone": "educational",
    },
    source=expert_source,
    metadata={
        "annotation_guideline": "physics_qa_standards_v2",
        "confidence": "high",
    },
)

print("Logged behavioral expectation with metadata.")

---
## 5 — Retrieving, Updating, and Deleting Assessments

Both feedback and expectations are *assessments* under the hood. You can retrieve, update, or delete any assessment using its ID.

### Finding assessment IDs

Assessment IDs are returned when you log feedback/expectations, and are also visible when you retrieve a trace.

In [ ]:
# Retrieve the trace to see all its assessments
trace = mlflow.get_trace(trace_ids[0])

print(f"Trace: {trace.info.trace_id[:16]}...")
print(f"Assessments: {len(trace.data.assessments)}")
print()
for a in trace.data.assessments:
    print(f"  [{a.assessment_id[:16]}...]  {a.name} = {a.value}  (source: {a.source.source_type})")

In [ ]:
# Retrieve a specific assessment by ID
if trace.data.assessments:
    first_assessment = trace.data.assessments[0]
    retrieved = mlflow.get_assessment(
        trace_id=trace_ids[0],
        assessment_id=first_assessment.assessment_id,
    )
    print(f"Name: {retrieved.name}")
    print(f"Value: {retrieved.value}")
    print(f"Rationale: {retrieved.rationale}")
    print(f"Source: {retrieved.source.source_type} / {retrieved.source.source_id}")

In [ ]:
from mlflow.entities import Feedback

# Update an existing assessment — for example, correcting a reviewer's score
if trace.data.assessments:
    target = trace.data.assessments[0]

    updated = Feedback(
        name=target.name,
        value=target.value,
        rationale="Updated rationale: Verified by a second reviewer, confirmed correct.",
    )

    mlflow.update_assessment(
        trace_id=trace_ids[0],
        assessment_id=target.assessment_id,
        assessment=updated,
    )

    print(f"Updated assessment {target.assessment_id[:16]}...")

In [ ]:
# Delete an assessment (use with care)
# Let's log one specifically to delete
mlflow.log_feedback(
    trace_id=trace_ids[0],
    name="temporary_note",
    value="This is a draft annotation",
    source=human_source,
)

# Find and delete it
trace = mlflow.get_trace(trace_ids[0])
for a in trace.data.assessments:
    if a.name == "temporary_note":
        mlflow.delete_assessment(
            trace_id=trace_ids[0],
            assessment_id=a.assessment_id,
        )
        print(f"Deleted assessment: {a.name}")
        break

---
## 6 — Overriding Automated Feedback

`mlflow.override_feedback()` is designed for a common workflow: an LLM judge scores a trace, but a human reviewer disagrees and wants to correct it. The override preserves the original assessment while recording the correction.

This creates an audit trail — you can see what the judge said *and* what the human corrected it to.

In [ ]:
# Step 1: Log automated feedback (simulating an LLM judge)
mlflow.log_feedback(
    trace_id=trace_ids[1],
    name="accuracy",
    value=0.5,
    rationale="The response omits key details about the Calvin cycle.",
    source=AssessmentSource(
        source_type=AssessmentSourceType.LLM_JUDGE,
        source_id="openai:/gpt-4o-mini",
    ),
)
print("LLM judge scored accuracy at 0.5")

# Step 2: Human reviewer disagrees — the response is actually fine for the scope
trace = mlflow.get_trace(trace_ids[1])
for a in trace.data.assessments:
    if a.name == "accuracy" and a.source.source_type == "LLM_JUDGE":
        mlflow.override_feedback(
            trace_id=trace_ids[1],
            assessment_id=a.assessment_id,
            value=0.85,
            rationale=(
                "Override: The question asked for 'one paragraph' — omitting the Calvin cycle "
                "is appropriate for this scope. The response covers light reactions correctly."
            ),
            source=AssessmentSource(
                source_type=AssessmentSourceType.HUMAN,
                source_id="biology_expert@example.com",
            ),
        )
        print(f"Human override: accuracy corrected to 0.85")
        break

---
## 7 — Batch Annotation Patterns

In practice, you often need to annotate many traces at once — a reviewer works through a queue and labels each one. Here's a helper pattern for batch annotation.

In [ ]:
def annotate_batch(annotations: list[dict], annotator_id: str) -> dict:
    """Annotate multiple traces with feedback.

    Each annotation dict should have:
        - trace_id: str
        - name: str (feedback dimension)
        - value: any (the assessment)
        - rationale: str (optional)
    """
    source = AssessmentSource(
        source_type=AssessmentSourceType.HUMAN,
        source_id=annotator_id,
    )

    results = {"success": 0, "failed": 0}
    for item in annotations:
        try:
            mlflow.log_feedback(
                trace_id=item["trace_id"],
                name=item["name"],
                value=item["value"],
                rationale=item.get("rationale", ""),
                source=source,
            )
            results["success"] += 1
        except Exception as e:
            print(f"Failed to annotate {item['trace_id'][:16]}...: {e}")
            results["failed"] += 1

    return results

In [ ]:
# Simulate a reviewer working through a queue
review_queue = [
    {"trace_id": trace_ids[0], "name": "helpfulness", "value": 5, "rationale": "Direct and accurate."},
    {"trace_id": trace_ids[1], "name": "helpfulness", "value": 4, "rationale": "Good but could be more thorough."},
    {"trace_id": trace_ids[2], "name": "helpfulness", "value": 4, "rationale": "Correct but a bit long."},
]

results = annotate_batch(review_queue, annotator_id="reviewer_alice@example.com")
print(f"Batch complete: {results['success']} succeeded, {results['failed']} failed.")

In [ ]:
def annotate_batch_expectations(expectations: list[dict], annotator_id: str) -> dict:
    """Annotate multiple traces with ground-truth expectations.

    Each expectation dict should have:
        - trace_id: str
        - name: str
        - value: any (the ground truth)
        - metadata: dict (optional)
    """
    source = AssessmentSource(
        source_type=AssessmentSourceType.HUMAN,
        source_id=annotator_id,
    )

    results = {"success": 0, "failed": 0}
    for item in expectations:
        try:
            mlflow.log_expectation(
                trace_id=item["trace_id"],
                name=item["name"],
                value=item["value"],
                source=source,
                metadata=item.get("metadata"),
            )
            results["success"] += 1
        except Exception as e:
            print(f"Failed to annotate {item['trace_id'][:16]}...: {e}")
            results["failed"] += 1

    return results


# Label ground truth for multiple traces
ground_truth = [
    {
        "trace_id": trace_ids[0],
        "name": "expected_answer",
        "value": "Paris",
    },
    {
        "trace_id": trace_ids[1],
        "name": "expected_facts",
        "value": ["light energy", "carbon dioxide", "water", "glucose", "oxygen"],
    },
    {
        "trace_id": trace_ids[2],
        "name": "expected_facts",
        "value": ["zeroth law", "first law", "second law", "third law"],
        "metadata": {"note": "Some sources include the zeroth law"},
    },
]

results = annotate_batch_expectations(ground_truth, annotator_id="physics_prof@example.com")
print(f"Expectations logged: {results['success']} succeeded, {results['failed']} failed.")

---
## 8 — Span-Level Feedback

By default, feedback attaches to the **trace** (the entire execution). But you can also attach feedback to a specific **span** within the trace — for example, rating the quality of a retrieval step separately from the final answer.

Pass the `span_id` parameter to target a specific span.

In [ ]:
# Get a trace and find a span to annotate
trace = mlflow.get_trace(trace_ids[0])
spans = trace.data.spans

print(f"Trace has {len(spans)} span(s):")
for s in spans:
    print(f"  {s.name} (span_id: {s.span_id[:16]}..., type: {s.span_type})")

In [ ]:
# Attach feedback to a specific span
if spans:
    target_span = spans[0]

    mlflow.log_feedback(
        trace_id=trace_ids[0],
        span_id=target_span.span_id,
        name="latency_acceptable",
        value=True,
        rationale="Span completed within acceptable time.",
        source=AssessmentSource(
            source_type=AssessmentSourceType.CODE,
            source_id="latency_monitor_v1",
        ),
    )

    print(f"Span-level feedback logged on span '{target_span.name}'.")

In [ ]:
# Span-level expectations are useful for RAG pipelines —
# e.g., the retrieval span should have returned specific documents
if spans:
    mlflow.log_expectation(
        trace_id=trace_ids[0],
        span_id=spans[0].span_id,
        name="expected_model_called",
        value="gemini-3.1-flash-lite-preview",
        source=AssessmentSource(
            source_type=AssessmentSourceType.CODE,
            source_id="config_validator_v1",
        ),
    )

    print(f"Span-level expectation logged on span '{spans[0].name}'.")

---
## 9 — Using Metadata for Context

The `metadata` parameter on both `log_feedback()` and `log_expectation()` lets you attach arbitrary key-value context. This is useful for tracking annotation workflows, linking to guidelines, or recording confidence levels.

In [ ]:
mlflow.log_feedback(
    trace_id=trace_ids[2],
    name="expert_review",
    value=True,
    rationale="All three laws are correctly stated.",
    source=AssessmentSource(
        source_type=AssessmentSourceType.HUMAN,
        source_id="physics_reviewer@example.com",
    ),
    metadata={
        "review_session": "2026-04-14-batch-1",
        "guideline_version": "physics_qa_v2",
        "confidence": "high",
        "time_spent_seconds": 45,
    },
)

print("Logged feedback with metadata.")

---
## 10 — Inspecting Everything on a Trace

Let's pull up one of our heavily annotated traces and see all the assessments together.

In [ ]:
trace = mlflow.get_trace(trace_ids[0])

print(f"Trace: {trace.info.trace_id}")
print(f"Total assessments: {len(trace.data.assessments)}")
print()

for a in trace.data.assessments:
    span_info = f" (span: {a.span_id[:12]}...)" if a.span_id else ""
    print(f"  {a.name}")
    print(f"    Value: {a.value}")
    print(f"    Source: {a.source.source_type} / {a.source.source_id}")
    if a.rationale:
        print(f"    Rationale: {a.rationale[:80]}..." if len(str(a.rationale)) > 80 else f"    Rationale: {a.rationale}")
    if span_info:
        print(f"    Span: {a.span_id}")
    print()

---
## Quick Reference

### Core APIs

| Function | Purpose |
|---|---|
| `mlflow.log_feedback(trace_id, name, value, source, rationale)` | Record a quality assessment |
| `mlflow.log_expectation(trace_id, name, value, source)` | Record ground truth |
| `mlflow.get_assessment(trace_id, assessment_id)` | Retrieve a specific assessment |
| `mlflow.update_assessment(trace_id, assessment_id, assessment)` | Modify an existing assessment |
| `mlflow.delete_assessment(trace_id, assessment_id)` | Remove an assessment |
| `mlflow.override_feedback(trace_id, assessment_id, value, source)` | Correct automated feedback |

### Assessment Sources

```python
from mlflow.entities import AssessmentSource
from mlflow.entities.assessment_source import AssessmentSourceType

# Human reviewer
AssessmentSource(source_type=AssessmentSourceType.HUMAN, source_id="alice@example.com")

# Programmatic rule
AssessmentSource(source_type=AssessmentSourceType.CODE, source_id="checker_v1")

# LLM judge
AssessmentSource(source_type=AssessmentSourceType.LLM_JUDGE, source_id="openai:/gpt-4o")
```

### Value Types

| Type | Example | Best for |
|---|---|---|
| `bool` | `True` / `False` | Pass/fail, thumbs up/down |
| `int` | `4` | Star ratings |
| `float` | `0.85` | Confidence scores |
| `str` | `"excellent"` | Categorical labels |
| `dict` | `{"accuracy": 0.9, ...}` | Multi-dimensional reviews |
| `list` | `["fact1", "fact2"]` | Expected facts/documents |

### Feedback vs Expectations

| | `log_feedback()` | `log_expectation()` |
|---|---|---|
| Evaluates | What the model *did* | What the model *should have done* |
| Used for | Quality scores, reviews | Ground truth, eval datasets |
| Typical source | Any (human, code, LLM) | Usually `HUMAN` |